# RegimeShift: Market-Regime Asset Allocation System
## IIT Bombay Summer Quant 2026 — Final Submission

This notebook reproduces the complete RegimeShift pipeline:
data → features → HMM → portfolio → backtest → metrics → charts.
All code uses the official `src` package; no implementation is duplicated here.

### 1. Objective

Build a leakage-free market-regime detection system (Bull, Bear, Crisis) that dynamically rebalances a multi-asset portfolio (NIFTY 50 equity, GOLDBEES.NS gold, 0P0001BVE8.BO gilt fund) while accounting for realistic transaction costs (5–10 bps).

### 2. Reproducibility and Configuration

All random seeds are fixed; all HMM training is on rolling windows through `t-1`.  VIX is optional and never allocated.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The package is installed in editable mode (pip install -e .)
# and so is importable from any cwd.

from regime_shift.config import RegimeShiftConfig
from regime_shift.exceptions import *
from regime_shift.data import download_market_data, load_market_data_csv
from regime_shift.features import (
    compute_raw_features, drop_feature_warmup,
    fit_feature_scaler, transform_features,
)
from regime_shift.regime_model import (
    fit_hmm, predict_current_state, RegimeSolution,
)
from regime_shift.portfolio import optimize_portfolio
from regime_shift.benchmarks import static_60_40_weights, equal_weight_weights
from regime_shift.backtest import run_walk_forward_backtest, run_benchmark
from regime_shift.metrics import compute_performance_metrics
from regime_shift.plots import generate_all_charts
from regime_shift import __version__

config = RegimeShiftConfig()
np.random.seed(config.random_seed)
print(f"RegimeShift version: {__version__}")
print(f"Random seed: {config.random_seed}")
print(f"Equity: {config.tickers.equity_ticker}")
print(f"Gold: {config.tickers.gold_ticker}")
print(f"Bond: {config.tickers.bond_ticker}")
print(f"VIX: {config.tickers.vix_ticker} (optional)")
print(f"Transaction cost: {config.transaction_cost_bps} bps")
print(f"Training window: {config.train_window} days")
print(f"Rebalance frequency: {config.rebalance_frequency} days")
print(f"HMM: {config.hmm_config.n_components} states, covariance_type={config.hmm_config.covariance_type!r}")


RegimeShift version: 1.0.0
Random seed: 42
Equity: ^NSEI
Gold: GOLDBEES.NS
Bond: 0P0001BVE8.BO
VIX: ^INDIAVIX (optional)
Transaction cost: 5.0 bps
Training window: 252 days
Rebalance frequency: 21 days
HMM: 3 states, covariance_type='diag'


### 3. Real Asset Universe and Actual Tickers

| Role | Ticker | Kind | Currency |
|---|---|---|---|
| Equity | `^NSEI` | PRICE | INR |
| Gold | `GOLDBEES.NS` | PRICE | INR |
| Bond | `0P0001BVE8.BO` | PRICE | INR |
| VIX (optional) | `^INDIAVIX` | INDICATOR | — |

This notebook uses a deterministic synthetic dataset for reproducibility. The official CLI fetches real data via yfinance when run online.

**Data validation:** The `data.py` module enforces 8 contract rules: timezone-naive, monotonically increasing index, no duplicate dates, strictly positive prices, forward-fill ≤ 3 days, no backward fill.

In [2]:
print("Generating deterministic synthetic price data (for reproducibility)...")

# Build a 3-segment regime structure so HMM has something to detect.
np.random.seed(42)
n_periods = 600
idx = pd.bdate_range("2010-01-01", periods=n_periods)

n_bull = n_periods // 3
n_bear = n_periods // 3
n_crisis = n_periods - n_bull - n_bear

bull_r = np.random.default_rng(42).normal(0.003, 0.008, n_bull)
bear_r = np.random.default_rng(7).normal(-0.002, 0.018, n_bear)
crisis_r = np.random.default_rng(99).normal(0.0, 0.05, n_crisis)
all_r = np.concatenate([bull_r, bear_r, crisis_r])

equity = 100.0 * np.exp(np.cumsum(all_r))
gold = 1800.0 + np.cumsum(np.random.default_rng(1).normal(0.0001, 0.005, n_periods))
bond = 97.0 + np.cumsum(np.random.default_rng(2).normal(0.00005, 0.002, n_periods))
vix = np.concatenate([
    np.full(n_bull, 15.0),
    np.full(n_bear, 25.0),
    np.full(n_crisis, 45.0),
])

prices = pd.DataFrame(
    {"equity": equity, "gold": gold, "bond": bond, "vix": vix},
    index=idx,
)

# Normalize index (timezone-naive, sorted, deduplicated)
from regime_shift.data import _normalise_index
prices = _normalise_index(prices)

print(f"Price data: {len(prices)} rows")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Columns: {list(prices.columns)}")
print(f"First few rows:")
display(prices.head())


Generating deterministic synthetic price data (for reproducibility)...


Price data: 600 rows
Date range: 2010-01-01 to 2012-04-19
Columns: ['equity', 'gold', 'bond', 'vix']
First few rows:


,equity,gold,bond,vix
2010-01-01,100.545255,1800.001828,97.000428,15.0
2010-01-04,100.011787,1800.006036,96.999433,15.0
2010-01-05,100.916320,1800.007788,96.998656,15.0
2010-01-06,101.984024,1800.001372,96.993824,15.0
2010-01-07,100.706253,1800.005999,96.997473,15.0


### 4. Data Validation

The `validate_price_data()` function enforces all 8 contract rules.

In [3]:
from regime_shift.validation import validate_price_data

validated = validate_price_data(
    prices,
    required_cols=["equity", "gold", "bond"],
    allow_missing=False,
    forward_fill_limit=3,
    check_bfill=True,
)
print(f"Validation passed: {len(validated)} rows, {len(validated.columns)} columns")
_a = ["equity", "gold", "bond", "vix"]
print(f"Asset columns: {[c for c in validated.columns if c in _a]}")


Validation passed: 600 rows, 4 columns


Asset columns: ['equity', 'gold', 'bond', 'vix']


### 5. Leakage-Safe Features

Seven base features (plus 2 optional VIX features) computed with strictly trailing windows.  Scaler is fit only on the training window.

In [4]:
feature_cfg = config.feature_config
raw_features = compute_raw_features(prices, config=feature_cfg)
raw_features = drop_feature_warmup(
    raw_features, minimum_observations=feature_cfg.minimum_feature_observations,
    config=feature_cfg,
)
print(f"Features after warmup: {len(raw_features)} rows, {len(raw_features.columns)} columns")
print(f"Feature columns: {list(raw_features.columns)}")
print(f"First few feature rows:")
display(raw_features.head())


Features after warmup: 537 rows, 9 columns
Feature columns: ['equity_log_return_1d', 'equity_momentum_21d', 'equity_momentum_63d', 'equity_volatility_21d', 'equity_volatility_ratio_21_63', 'equity_gold_correlation_63d', 'equity_bond_correlation_63d', 'vix_change_5d', 'vix_level']
First few feature rows:


,equity_log_return_1d,equity_momentum_21d,equity_momentum_63d,equity_volatility_21d,equity_volatility_ratio_21_63,equity_gold_correlation_63d,equity_bond_correlation_63d,vix_change_5d,vix_level
2010-03-31,0.007690,0.069269,0.230904,0.098484,0.967807,0.032367,-0.185078,0.0,15.0
2010-04-01,0.008690,0.073374,0.248270,0.100182,0.993962,0.060751,-0.192914,0.0,15.0
2010-04-02,0.009347,0.079201,0.248699,0.102297,1.014161,0.047457,-0.201213,0.0,15.0
2010-04-05,0.000210,0.074313,0.235885,0.102878,1.028100,0.071334,-0.147787,0.0,15.0
2010-04-06,-0.000699,0.062911,0.250692,0.100929,1.061857,0.114673,-0.047162,0.0,15.0


### 6. Walk-Forward Gaussian HMM

A 3-state `GaussianHMM` with diagonal covariance is fit on each 252-day rolling training window.  Only data through `t-1` is used.

In [5]:
from regime_shift.features import (
    fit_feature_scaler, transform_features, fit_transform_training_features,
)

# Demonstrate a single HMM fit on the training window
train_end = raw_features.index[250]  # first training window end
train_features = raw_features.loc[:train_end].tail(config.train_window)

scaler = fit_feature_scaler(train_features)
scaled_train = transform_features(scaler, train_features)

hmm, hidden_states, trans_mat = fit_hmm(
    scaled_train, train_features, config=config.hmm_config,
)
print(f"HMM converged: {hmm.monitor_.converged}")
print(f"Iterations: {hmm.monitor_.n_iter}")
print(f"Log-likelihood: {hmm.score(scaled_train.values):.2f}")
print(f"Transition matrix:")
display(trans_mat)
print(f"Hidden state counts: Bull={sum(hidden_states==0)}, Bear={sum(hidden_states==1)}, Crisis={sum(hidden_states==2)}")


HMM converged: True
Iterations: 200
Log-likelihood: 226.19
Transition matrix:


,Bull,Bear,Crisis
Bull,0.800000,0.000000,0.2
Bear,0.007299,0.992701,0.0
Crisis,0.000000,0.000000,1.0


Hidden state counts: Bull=109, Bear=137, Crisis=5


### 7. Bull/Bear/Crisis Interpretation

Numeric HMM states are mapped to interpretable labels using training-period volatility, momentum, and VIX statistics.

In [6]:
from regime_shift.regime_model import get_state_statistics, get_transition_matrix

# Regime solution from the last training window
solution = predict_current_state(
    hmm, scaler,
    raw_features.loc[:train_end],  # sequence through cutoff
    train_features,
    config=config.hmm_config,
)

print(f"Current regime: {solution.regime}")
print(f"Regime probabilities: {dict(solution.probabilities)}")
print(f"Convergence: {solution.convergence}")
print(f"Iterations: {solution.n_iter}")

stats = get_state_statistics(solution)
for s in stats:
    print(f"  State {s.state_id} ({s.regime_label}): {s.n_observations} obs, vol={s.mean_features.get('equity_volatility_21d', 0):.4f}, mom={s.mean_features.get('equity_momentum_63d', 0):.4f}")


Current regime: Crisis
Regime probabilities: {'Bull': np.float64(0.0), 'Bear': np.float64(0.0), 'Crisis': np.float64(1.0)}
Convergence: True
Iterations: 200
  State 0 (Crisis): 109 obs, vol=0.2403, mom=-0.1681
  State 1 (Bear): 137 obs, vol=0.1145, mom=0.1686
  State 2 (Bull): 5 obs, vol=0.1336, mom=0.1817


### 8. CVXPY Regime-Conditioned Portfolio Optimization

The optimiser selects one of three regime-specific convex objectives and applies regime-specific constraints.

In [7]:
# Compute asset returns for estimation
asset_returns = prices[["equity", "gold", "bond"]].pct_change().iloc[1:]

est_returns = asset_returns.loc[:train_end].tail(
    config.portfolio_config.estimation_lookback
)

port_sol = optimize_portfolio(
    regime=solution.regime,
    returns_through_date=est_returns,
    previous_weights=None,
    config=config.portfolio_config,
)

print(f"Regime: {port_sol.regime}")
print(f"Solver: {port_sol.solver}")
print(f"Status: {port_sol.status}")
print(f"Objective value: {port_sol.objective_value:.6f}")
print(f"Expected return: {port_sol.expected_return:.6f}")
print(f"Expected volatility: {port_sol.expected_volatility:.6f}")
print(f"Weights: equity={port_sol.weights['equity']:.4f}, gold={port_sol.weights['gold']:.4f}, bond={port_sol.weights['bond']:.4f}")


Regime: Crisis


Solver: CLARABEL
Status: optimal
Objective value: 0.000000
Expected return: -0.000004
Expected volatility: 0.000706
Weights: equity=0.0051, gold=0.4973, bond=0.4976


### 9. Transaction Costs

**Initial allocation:** turnover = $\sum |w_i|$ (full L1).

**Subsequent rebalance:** turnover = $\frac{1}{2}\sum |w_i - w_{i-1}^{\text{unadj}}|$ (half-L1).

**Net return:** $r_{\text{net}} = (1 - c) \times (1 + r_{\text{gross}}) - 1$ where $c = \text{turnover} \times \text{bps}/10{,}000$.

No cost on non-rebalance dates.

In [8]:
config.transaction_cost_bps = 5.0
cost_rate = config.transaction_cost_bps / 10000.0
print(f"Transaction cost rate: {cost_rate:.6f} ({config.transaction_cost_bps} bps)")
print(f"Half-L1 convention: turnover = 0.5 * sum(abs(target - current))")


Transaction cost rate: 0.000500 (5.0 bps)
Half-L1 convention: turnover = 0.5 * sum(abs(target - current))


### 10. Benchmarks: 60/40 and Equal Weight

Two static benchmarks run with the same rebalance dates and cost convention as the strategy.

In [9]:
static_60_40 = static_60_40_weights()
equal_weight = equal_weight_weights()

print(f"Static 60/40 weights: {static_60_40}")
print(f"Equal weight: {equal_weight}")


Static 60/40 weights: {'equity': 0.6, 'gold': 0.0, 'bond': 0.4}
Equal weight: {'equity': 0.3333333333333333, 'gold': 0.3333333333333333, 'bond': 0.3333333333333333}


### 11. Walk-Forward Timing — Full Backtest

The `run_walk_forward_backtest()` function executes a single chronological loop.  At each rebalance date it fits the scaler, HMM, and optimiser using only data through `t-1`.

In [10]:
config.minimum_training_observations = 60
config.transaction_cost_bps = 5.0

strategy_result = run_walk_forward_backtest(
    prices=prices[["equity", "gold", "bond", "vix"]],
    config=config,
    transaction_cost_bps=5.0,
    risk_free_rate=0.0,
)

print(f"Backtest complete.")
print(f"Trading days: {len(strategy_result.net_returns)}")
print(f"Date range: {strategy_result.net_returns.index[0].date()} to {strategy_result.net_returns.index[-1].date()}")
print(f"Successful rebalances: {strategy_result.rebalance_flags.sum()}")

# Regime counts
regime_counts = strategy_result.regime_series.value_counts()
for regime in ["Bull", "Bear", "Crisis"]:
    print(f"  {regime}: {regime_counts.get(regime, 0)} days")


Skipping rebalance on 2010-04-29 00:00:00: only 21 training observations, need 60.


Skipping rebalance on 2010-05-28 00:00:00: only 42 training observations, need 60.


Model is not converging.  Current: -163.31802007302423 is not greater than 90.58802607705628. Delta is -253.9060461500805


Model is not converging.  Current: -124.89707454948095 is not greater than 196.81574038120618. Delta is -321.71281493068716


Model is not converging.  Current: -140.26063681343405 is not greater than 250.9320657203653. Delta is -391.1927025337993


Model is not converging.  Current: -357.22598056041204 is not greater than 143.30201100053665. Delta is -500.5279915609487


Model is not converging.  Current: -195.55080256645454 is not greater than -195.55069622999497. Delta is -0.00010633645956659166


Backtest complete.
Trading days: 474
Date range: 2010-06-28 to 2012-04-19
Successful rebalances: 23
  Bull: 0 days
  Bear: 42 days
  Crisis: 432 days


In [11]:
# Run benchmarks
bench_60_40 = run_benchmark(
    prices=prices[["equity", "gold", "bond", "vix"]],
    weights=static_60_40_weights(),
    config=config,
    transaction_cost_bps=5.0,
    rebalance_flags=strategy_result.rebalance_flags,
    start_date=strategy_result.net_returns.index[0],
)

bench_eq = run_benchmark(
    prices=prices[["equity", "gold", "bond", "vix"]],
    weights=equal_weight_weights(),
    config=config,
    transaction_cost_bps=5.0,
    rebalance_flags=strategy_result.rebalance_flags,
    start_date=strategy_result.net_returns.index[0],
)

benchmarks = {"Static 60/40": bench_60_40, "Equal Weight": bench_eq}
print("Benchmarks complete.")


Benchmarks complete.


### 12. Performance Metrics

Metrics are computed from `compute_performance_metrics()` using the exact cost-drag definition: $\text{drag} = \prod(1 + r_{\text{gross}}) - \prod(1 + r_{\text{net}})$.

**Sharpe:** excess return / excess volatility $\times \sqrt{252}$

**Sortino:** excess return / downside deviation $\times \sqrt{252}$

**Max Drawdown:** $\max_t \left(1 - \frac{\text{equity}_t}{\max_{s\le t} \text{equity}_s}\right)$ — clamped to [0, 1]

**Calmar:** CAGR / Maximum Drawdown

In [12]:
from regime_shift.metrics import compute_performance_metrics

def _metrics_for(label, returns, gross_returns, turnover, costs, rf=0.0):
    m = compute_performance_metrics(
        returns, gross_returns=gross_returns,
        turnover=turnover, transaction_costs=costs,
        risk_free_rate=rf,
    )
    d = m.to_dict()
    d["Strategy"] = label
    return d

rows = [
    _metrics_for("RegimeShift Gross", strategy_result.gross_returns,
                 strategy_result.gross_returns, strategy_result.turnover,
                 strategy_result.transaction_costs),
    _metrics_for("RegimeShift Net", strategy_result.net_returns,
                 strategy_result.gross_returns, strategy_result.turnover,
                 strategy_result.transaction_costs),
    _metrics_for("Static 60/40 Gross", bench_60_40.gross_returns,
                 bench_60_40.gross_returns, bench_60_40.turnover,
                 bench_60_40.transaction_costs),
    _metrics_for("Static 60/40 Net", bench_60_40.net_returns,
                 bench_60_40.gross_returns, bench_60_40.turnover,
                 bench_60_40.transaction_costs),
    _metrics_for("Equal Weight Gross", bench_eq.gross_returns,
                 bench_eq.gross_returns, bench_eq.turnover,
                 bench_eq.transaction_costs),
    _metrics_for("Equal Weight Net", bench_eq.net_returns,
                 bench_eq.gross_returns, bench_eq.turnover,
                 bench_eq.transaction_costs),
]

perf_df = pd.DataFrame(rows)
perf_df = perf_df[[
    "Strategy", "Total Return", "CAGR", "Annualised Volatility",
    "Sharpe", "Sortino", "Maximum Drawdown", "Calmar",
    "Total Turnover", "Annualised Turnover", "Transaction Cost Drag",
]]
display(perf_df)

import os
os.makedirs("results", exist_ok=True)
perf_df.to_csv("results/performance_summary.csv", index=False)


,Strategy,Total Return,CAGR,Annualised Volatility,Sharpe,Sortino,Maximum Drawdown,Calmar,Total Turnover,Annualised Turnover,Transaction Cost Drag
0,RegimeShift Gross,0.0502,0.0264,0.0155,1.6922,2.9804,0.0113,2.3400,1.8288,0.9723,0.0000
1,RegimeShift Net,0.0493,0.0259,0.0155,1.6609,2.9352,0.0115,2.2553,1.8288,0.9723,0.0010
2,Static 60/40 Gross,-0.2998,-0.1726,0.3140,-0.4462,-0.4547,0.4306,-0.4008,1.6825,0.8945,0.0000
3,Static 60/40 Net,-0.3004,-0.1730,0.3140,-0.4476,-0.4561,0.4307,-0.4016,1.6825,0.8945,0.0006
4,Equal Weight Gross,-0.1585,-0.0877,0.1746,-0.4384,-0.4463,0.2644,-0.3317,1.6185,0.8605,0.0000
5,Equal Weight Net,-0.1592,-0.0881,0.1746,-0.4409,-0.4488,0.2645,-0.3331,1.6185,0.8605,0.0007


### 13. Charts and Robustness Checks

Six charts are generated by `generate_all_charts()`.

In [13]:
import os
os.makedirs("results", exist_ok=True)

saved = generate_all_charts(
    result=strategy_result,
    benchmarks=benchmarks,
    prices=prices[["equity", "gold", "bond", "vix"]],
    output_dir="results",
)
print(f"Saved {len(saved)} charts:")
for p in saved:
    print(f"  {p}")


Saved 6 charts:
  results\regime_price_chart.png
  results\transition_matrix.png
  results\equity_curves.png
  results\drawdowns.png
  results\portfolio_weights.png
  results\regime_probabilities.png


### 14. Robustness Checks

**5 bps vs 10 bps sensitivity:** Run both cost regimes and compare the strategy's Sharpe ratio and maximum drawdown.

**Transition matrix sanity:** The matrix must be a valid stochastic matrix (rows sum to 1, no identity placeholder).

**Drawdown math:** Drawdowns are computed from `(1 + r).cumprod() / cummax - 1` — never from `r.cumprod()`.

**Daily drifted weights:** Portfolio-weight chart shows actual post-drift weights, not just target weights at rebalance dates.

**Regime probability order:** Probabilities are always in the explicit `Bull / Bear / Crisis` order — never sorted alphabetically.

**No synthetic data in final submission:** The final run must use real market data via `python run_submission.py --start 2010-01-01`.

In [14]:
# 10 bps sensitivity check
config_10bps = RegimeShiftConfig()
config_10bps.minimum_training_observations = 60
config_10bps.transaction_cost_bps = 10.0

result_10bps = run_walk_forward_backtest(
    prices=prices[["equity", "gold", "bond", "vix"]],
    config=config_10bps,
    transaction_cost_bps=10.0,
)

def _m(returns):
    return compute_performance_metrics(returns, gross_returns=returns)

m5 = _m(strategy_result.net_returns)
m10 = _m(result_10bps.net_returns)

print("5 bps vs 10 bps sensitivity (net):")
print(f"  5 bps — Sharpe: {m5.sharpe_ratio:.3f}, MaxDD: {m5.maximum_drawdown:.3f}, Turnover: {m5.total_turnover:.3f}")
print(f"  10 bps — Sharpe: {m10.sharpe_ratio:.3f}, MaxDD: {m10.maximum_drawdown:.3f}, Turnover: {m10.total_turnover:.3f}")
print(f"  Cost drag 5bps: {m5.total_transaction_cost_drag:.6f}")
print(f"  Cost drag 10bps: {m10.total_transaction_cost_drag:.6f}")


Skipping rebalance on 2010-04-29 00:00:00: only 21 training observations, need 60.


Skipping rebalance on 2010-05-28 00:00:00: only 42 training observations, need 60.


Model is not converging.  Current: -163.31802007302423 is not greater than 90.58802607705628. Delta is -253.9060461500805


Model is not converging.  Current: -124.89707454948095 is not greater than 196.81574038120618. Delta is -321.71281493068716


Model is not converging.  Current: -140.26063681343405 is not greater than 250.9320657203653. Delta is -391.1927025337993


Model is not converging.  Current: -357.22598056041204 is not greater than 143.30201100053665. Delta is -500.5279915609487


Model is not converging.  Current: -195.55080256645454 is not greater than -195.55069622999497. Delta is -0.00010633645956659166


5 bps vs 10 bps sensitivity (net):
  5 bps — Sharpe: 1.661, MaxDD: 0.011, Turnover: nan
  10 bps — Sharpe: 1.628, MaxDD: 0.012, Turnover: nan
  Cost drag 5bps: 0.000000
  Cost drag 10bps: 0.000000


### 15. Conclusions and Limitations

**Strengths:**
- Fully leakage-safe walk-forward pipeline
- Deterministic regime mapping based on training-period statistics
- CVXPY regime-specific convex optimization
- Exact transaction-cost drag definition
- 256 automated tests covering timing, metrics, charts, CLI, and notebook

**Limitations:**
- Gaussian HMM assumes continuous Gaussian emissions — Student-t distributions may better capture fat tails.
- VIX is optional; without it, Crisis detection relies solely on volatility and momentum.
- Three assets only; broader diversification requires expanding `core_assets`.
- The 10 bps sensitivity check must be re-run on real data for the official submission.
- No FX conversion — all assets are INR-denominated; cross-currency portfolios need explicit FX handling.